# Merging blocks

## Strategy
1. post-processing of existing susie rds object with get_susie_cs with converage = 0.5
2. pick the top1 variant in each credible set, ad get a list of variants per chrome.
3. using cyvcf2 to extract this entire matrix per chrome from vcf file.
4. Then compute their correlation by using plink2
5. if their is a correlation between different blocks, then we would like to merge them to be a big block.
 - the output would be a list of removed LD blocks and a list of new LD blocks with new range

In [228]:
library(susieR)
library(tidyverse)
library(data.table)

### step 1 and 2

In [50]:
files <- list.files("/mnt/vast/hpc/csg/ftp_lisanwanglab_sync/ftp_fgc_xqtl/projects/GWAS_Finemapping_Results/Bellenguez/",
                    pattern = "rds$", full.name = T)
dir.create("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/", recursive = T)
all_variants <- c()
for(chr in 1:22){
chr_variants <- c()
chr_files <- files %>% .[grep(paste0("chr",chr, "_"),.)]
    for(file in chr_files){
        susie_obj <- readRDS(file)
        ld_cs <- susie_get_cs(susie_obj, coverage = 0.5)$cs 
        ld_variants <- c()
        for(i in 1:length(ld_cs)){
            cs <- ld_cs[[i]]
            max_pos <- which.max((susie_obj$pip)[cs])
            ld_variants <- c(ld_variants, susie_obj$variants[max_pos]%>% sub("_",":",.))
        }
        chr_variants <- c(chr_variants, ld_variants)
        write.table(chr_variants, 
                    paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/chr",chr,"_variants_converage50"), 
                    row.names = F, col.names = F, quote = F)
    }
    all_variants <- c(all_variants, chr_variants)
}
write.table(all_variants, 
                    paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/ALL_variants_converage50"), 
                    row.names = F, col.names = F, quote = F)

Warning message in dir.create("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/", :
“'/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected' already exists”


### step3 extract variants for each chrom from bed file using plink

In [ ]:
mkdir -p ~/Work/pecotmr/merging_test/bed_subset && cd ~/Work/pecotmr/merging_test/bed_subset
module load Plink/1.9.10
for i in {1..22}
do
    plink --bfile /mnt/vast/hpc/csg/FunGen_xQTL/ROSMAP/genotype/ROSMAP_NIA_WGS.leftnorm.filtered.filtered \
        --extract /mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/chr${i}_variants_converage50 \
        --out ROSMAP_NIA_WGS.leftnorm.filtered.filtered.chr${i}.variants_converage50.extracted --memory 50000 \
        --make-bed
done


### step4 calculate ld correlation by each chrome using plink

In [ ]:
for i in {1..22}
do
    plink --bfile ROSMAP_NIA_WGS.leftnorm.filtered.filtered.chr${i}.variants_converage50.extracted \
        --r square \
        --out ROSMAP_NIA_WGS.leftnorm.filtered.filtered.chr${i}.variants_converage50.extracted.correlation
done

### step5 define correlation between ld blocks

In [ ]:
library(data.table)
merged_block <- data.frame()
ld_pos <- fread("/mnt/vast/hpc/csg/molecular_phenotype_calling/LD/1300_hg38_EUR_LD_blocks.tsv")
# if the correlation threshlod is 0.3
cor_thres <- 0.2

sort_and_join_blocks <- function(block, block_cor) {
  elements <- c(block, block_cor)
  return(paste(sort(elements), collapse = "_"))
}

are_blocks_adjacent <- function(block, block_cor) {
  num1 <- as.numeric(gsub("LDBlock", "", block))
  num2 <- as.numeric(gsub("LDBlock", "", block_cor))
  return(abs(num1 - num2) == 1)
}
# find the cor info on each chrom
for(chr in 1:22){
    cor <- fread(paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/bed_subset/ROSMAP_NIA_WGS.leftnorm.filtered.filtered.chr",chr,".variants_converage50.extracted.correlation.ld")) %>% as.data.frame
    cor_name <- fread(paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/bed_subset/ROSMAP_NIA_WGS.leftnorm.filtered.filtered.chr",chr,".variants_converage50.extracted.bim"))
    
    #add block info for each selected variant
    cor_name[, `#chr` := sub("^(chr[0-9XY]+):.*$", "\\1", V2)]
    cor_name[, block := ld_pos[cor_name, on = .(`#chr`, start <= V4, stop >= V4), x.ID, mult = "first"]]
    rownames(cor) <- colnames(cor) <- cor_name$V2
    
    # loop for each LD block                                   
    tmp_merged_block <- df <- data.frame()
    for (block_id in unique(cor_name$block)){
        vars_name <- cor_name[cor_name$block == block_id,]$V2

        #to check if there is a variant **outside of the LD block** on same chrom is highly correlated with that variant
        be_related_var <- rownames(cor)[rowSums(cor[, vars_name, drop=F] > cor_thres) > 0 ]
        be_related_df <- cor_name[cor_name$V2 %in% be_related_var,] %>% filter(block != block_id)
        # if so, add those two block info      
         if(nrow(be_related_df)>0) { 
         be_related_df$block_cor <- block_id
         df <- rbind(df, be_related_df)
        }
    }
                                                
    ## combine each LD block together for each chrom                                         
    if(nrow(df)>0) { 
        df$sorted_blocks <- mapply(sort_and_join_blocks, df$block, df$block_cor)
        df$is_adjacent <- mapply(are_blocks_adjacent, df$block, df$block_cor)
        # Remove duplicates
        tmp_merged_block <- df[!duplicated(df$sorted_blocks), c("V1","block","block_cor","sorted_blocks","is_adjacent")]
        colnames(tmp_merged_block)[1] <- "#chr"
        
    }
    merged_block <- rbind(merged_block, tmp_merged_block)                                        
                                            
}
    

                                            
write.table(merged_block, paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/correlation_",cor_thres,"_merged_block.txt"), sep = '\t', 
     row.names = F, quote = F)

# merging adjacent ones

adj_merged_block <- merged_block %>% filter(is_adjacent)

ld_removed <- ld_pos %>% filter(ID %in% adj_merged_block$block | ID %in% adj_merged_block$block_cor)

ld_remain <- ld_pos %>% filter(!(ID %in% adj_merged_block$block | ID %in% adj_merged_block$block_cor))

ld_new_blocks <-data.frame(
    `#chr` = adj_merged_block$`#chr`, 
    start = ld_pos %>% filter( ID %in% adj_merged_block$block_cor) %>% select(start),
    stop = ld_pos %>% filter( ID %in% adj_merged_block$block) %>% select(stop),
    ID = adj_merged_block$sorted_blocks, check.names = F
)
full_ld_merged_blocks <- rbind(ld_remain, ld_new_blocks)

write.table(ld_removed, paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/correlation_",cor_thres,"_hg38_removed_blocks.txt"), sep = '\t', 
     row.names = F, quote = F)
write.table(ld_new_blocks, paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/correlation_",cor_thres,"_hg38_new_merged_blocks.txt"), sep = '\t', 
     row.names = F, quote = F)
write.table(full_ld_merged_blocks, paste0("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/correlation_",cor_thres,"_hg38_full_ld_merged_blocks.txt"), sep = '\t', 
     row.names = F, quote = F)

### Packaging step3-5 to R function

In [16]:
library(susieR)
library(tidyverse)
library(data.table)
extract_variants <- function(bfile_path, bfile_prefix, variants_dir, plink_output_dir, plink_path = '/mnt/mfs/cluster/bin/plink1.9.10/plink') {
    message("extracting variants from bed files")
    for (i in 1:22) {
        variant_file <- list.files(variants_dir, full.name = T) %>% .[grep(paste0("chr", i, "(\\D)"), .)]
        output_file <- paste0(plink_output_dir, "/", bfile_prefix,".", basename(variant_file))
        command <- paste(plink_path, " --bfile", bfile_path,
                         "--extract", variant_file,
                         "--out", output_file,
                         "--make-bed --memory 50000")
        system(command)
    }
}

calculate_ld_correlation <- function(plink_output_dir, bfile_prefix, plink_path = '/mnt/mfs/cluster/bin/plink1.9.10/plink') {
    message("calculate the LD from the variants")
    for (i in 1:22) {
        bfile <- list.files(plink_output_dir,pattern = "bed", full.name = T) %>% .[grep(paste0("chr", i, "(\\D)"), .)] %>% gsub(".bed","",.) 
        output_file <- paste0(bfile, ".correlation")
        command <- paste(plink_path, " --bfile", bfile,
                         "--r square --out", output_file)
        system(command, intern = TRUE)
    }
}

define_correlation_blocks <- function(plink_output_dir, ld_pos_path, cor_thres, bfile_prefix) {
    message("defining the correlation among LD blocks")
    library(data.table)
    merged_block <- data.frame()
    ld_pos <- fread(ld_pos_path)

    for (chr in 1:22) {
        cor_file <- list.files(plink_output_dir,pattern = "ld", full.name = T) %>% .[grep(paste0("chr", chr, "(\\D)"), .)]
        cor <- fread(cor_file) %>% as.data.frame
        cor_name <- fread(list.files(plink_output_dir,pattern = "bim", full.name = T) %>% .[grep(paste0("chr", chr, "(\\D)"), .)])

        #add block info for each selected variant
        cor_name[, `#chr` := sub("^(chr[0-9XY]+):.*$", "\\1", V2)]
        cor_name[, block := ld_pos[cor_name, on = .(`#chr`, start <= V4, stop >= V4), x.ID, mult = "first"]]
        rownames(cor) <- colnames(cor) <- cor_name$V2

        # loop for each LD block                                   
        tmp_merged_block <- df <- data.frame()
        for (block_id in unique(cor_name$block)){
            vars_name <- cor_name[cor_name$block == block_id,]$V2

            #to check if there is a variant **outside of the LD block** on same chrom is highly correlated with that variant
            be_related_var <- rownames(cor)[rowSums(cor[, vars_name, drop=F] > cor_thres) > 0 ]
            be_related_df <- cor_name[cor_name$V2 %in% be_related_var,] %>% filter(block != block_id)
            # if so, add those two block info      
             if(nrow(be_related_df)>0) { 
             be_related_df$block_cor <- block_id
             df <- rbind(df, be_related_df)
            }
        }

        ## combine each LD block together for each chrom                                         
        if(nrow(df)>0) { 
            df$sorted_blocks <- mapply(sort_and_join_blocks, df$block, df$block_cor)
            df$is_adjacent <- mapply(are_blocks_adjacent, df$block, df$block_cor)
            # Remove duplicates
            tmp_merged_block <- df[!duplicated(df$sorted_blocks), c("V1","block","block_cor","sorted_blocks","is_adjacent")]
            colnames(tmp_merged_block)[1] <- "#chr"

        }
        merged_block <- rbind(merged_block, tmp_merged_block)                                        

    }
    return(merged_block)
}

merge_adjacent_blocks <- function(merged_block, ld_pos_path, output_dir, cor_thres) {
    message("merging the adjacent correlated LD blocks")
    adj_merged_block <- merged_block %>% filter(is_adjacent)
    ld_pos <- fread(ld_pos_path)

    ld_removed <- ld_pos %>% filter(ID %in% adj_merged_block$block | ID %in% adj_merged_block$block_cor)
    ld_remain <- ld_pos %>% filter(!(ID %in% adj_merged_block$block | ID %in% adj_merged_block$block_cor))
    ld_new_blocks <-data.frame(
        `#chr` = paste0("chr",adj_merged_block$`#chr`), 
        start = ld_pos %>% filter( ID %in% adj_merged_block$block_cor) %>% select(start),
        stop = ld_pos %>% filter( ID %in% adj_merged_block$block) %>% select(stop),
        ID = adj_merged_block$sorted_blocks, check.names = F
    )
    full_ld_merged_blocks <- rbind(ld_remain, ld_new_blocks)
    final <- list(ld_removed = ld_removed,
                 ld_new_blocks = ld_new_blocks,
                 full_ld_merged_blocks = full_ld_merged_blocks)
    saveRDS(final, file.path(output_dir,paste0("corelation_", cor_thres,"_",basename(ld_pos_path)%>%sub("\\.[^.]*$", "", .), "_merged.rds")))
}

process_variants <- function(bfile_path, variants_dir, ld_pos_path, cor_thres, output_dir = '.') {
    # Extracting the basename prefix from bfile_path
    plink_output_dir <- paste0(output_dir,"/plink/") 
    dir.create(plink_output_dir, recursive = T)
    bfile_basename <- basename(bfile_path)
    #bfile_prefix <- sub("\\..*$", "", bfile_basename)  # Removes the extension and keeps the prefix
    bfile_prefix <- bfile_basename
    extract_variants(bfile_path, bfile_prefix, variants_dir, plink_output_dir)
    calculate_ld_correlation(plink_output_dir, bfile_prefix)
    merged_block <- define_correlation_blocks(plink_output_dir, ld_pos_path, cor_thres, bfile_prefix)
    merge_adjacent_blocks(merged_block, ld_pos_path, output_dir, cor_thres)
}


# Define helper functions
sort_and_join_blocks <- function(block, block_cor) {
elements <- c(block, block_cor)
return(paste(sort(elements), collapse = "_"))
}

are_blocks_adjacent <- function(block, block_cor) {
  num1 <- as.numeric(gsub("LDBlock", "", block))
  num2 <- as.numeric(gsub("LDBlock", "", block_cor))
  return(abs(num1 - num2) == 1)
}

In [17]:
process_variants(bfile_path = "/mnt/vast/hpc/csg/FunGen_xQTL/ROSMAP/genotype/ROSMAP_NIA_WGS.leftnorm.filtered.filtered", 
                 variants_dir = "/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/variants_selected/", 
                 ld_pos_path = "/mnt/vast/hpc/csg/molecular_phenotype_calling/LD/1300_hg38_EUR_LD_blocks.tsv", 
                 cor_thres = 0.2, 
                 output_dir = "/home/rf2872/Work/pecotmr/merging_test/test")


Warning message in dir.create(plink_output_dir, recursive = T):
“'/home/rf2872/Work/pecotmr/merging_test/test/plink' already exists”
calculate the LD from the variants

defining the correlation among LD blocks

merging the adjacent correlated LD blocks



#### results of R function

In [18]:
tmp <- readRDS("/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/test/corelation_0.2_1300_hg38_EUR_LD_blocks_merged.rds")

In [20]:
str(tmp)

List of 3
 $ ld_removed           :Classes ‘data.table’ and 'data.frame':	18 obs. of  4 variables:
  ..$ #chr : chr [1:18] "chr3" "chr3" "chr3" "chr3" ...
  ..$ start: int [1:18] 47545024 52179438 152189686 154360351 28390030 30136357 66619047 67338402 95543074 98297733 ...
  ..$ stop : int [1:18] 52179438 54975538 154360351 156292136 30136357 31604117 67338402 68812188 98297733 101733715 ...
  ..$ ID   : chr [1:18] "LDBlock246" "LDBlock247" "LDBlock294" "LDBlock295" ...
  ..- attr(*, ".internal.selfref")=<externalptr> 
 $ ld_new_blocks        :'data.frame':	9 obs. of  4 variables:
  ..$ #chr : chr [1:9] "chr3" "chr3" "chr6" "chr6" ...
  ..$ start: int [1:9] 47545024 152189686 28390030 66619047 95543074 32921536 119137214 61125463 29423003
  ..$ stop : int [1:9] 54975538 156292136 31604117 68812188 101733715 40805126 124768254 66680537 32269392
  ..$ ID   : chr [1:9] "LDBlock246_LDBlock247" "LDBlock294_LDBlock295" "LDBlock518_LDBlock519" "LDBlock537_LDBlock538" ...
 $ full_ld_merged_bl

In [19]:
tmp$ld_new_blocks

#chr,start,stop,ID
<chr>,<int>,<int>,<chr>
chr3,47545024,54975538,LDBlock246_LDBlock247
chr3,152189686,156292136,LDBlock294_LDBlock295
chr6,28390030,31604117,LDBlock518_LDBlock519
chr6,66619047,68812188,LDBlock537_LDBlock538
chr7,95543074,101733715,LDBlock637_LDBlock638
chr12,32921536,40805126,LDBlock961_LDBlock962
chr12,119137214,124768254,LDBlock1000_LDBlock1001
chr15,61125463,66680537,LDBlock1123_LDBlock1124
chr22,29423003,32269392,LDBlock1349_LDBlock1350


In [28]:
write.table(tmp$ld_new_blocks, "/mnt/vast/hpc/csg/rf2872/Work/pecotmr/merging_test/test/corelation_0.2_1300_hg38_EUR_LD_blocks_merged.new_blocks.tsv", row.names = F, quote = F, sep = '\t')

### test Tosin'd code to extract variants

In [2]:
from cyvcf2 import VCF
import numpy as np
from math import nan
from collections import Counter
import pandas as pd
import sys


# gets dosages (which pass filter criteria) from VCF file object
def get_dosages(vcf_obj, maf_min = 0.05, mac_min = 5, msng_min = 0.01,
                set_samples=None):
    arr = []
    var_names = []
    for var in vcf_obj:
        if len(var.ALT) != 1:
            print("Skipping multi-allelic variant")
        dosage = [sum(x[0:2]) for x in [[nan if x1 == -1 else x1 for x1 in x0] for x0 in var.genotypes]]
        counts = {0:sum([2 - x for x in dosage]), 1:sum(dosage)}
        num_samp_non_na = sum(counts.values())/2
        mac = min(counts[0], counts[1])
        maf = mac / num_samp_non_na/ 2
        nan_count = np.sum(np.isnan(dosage))
        msng_rate = nan_count / (num_samp_non_na + nan_count)
        if (maf < maf_min) or (mac < mac_min) or (msng_rate > msng_min):
            continue
        arr.append(dosage)
        var_names.append("_".join([var.CHROM, str(var.POS), var.REF, var.ALT[0]]))
    return pd.DataFrame(arr, index=var_names, columns = set_samples)


# returns output of dosages on a per LD block basis
def get_ld_block_dosages(vcf_map, ld_block_file):
    out = {}
    with open(ld_block_file) as f:
        for line in f:
            elems = line.split()
            elems[-1] = elems[-1].strip()
            chrm = elems[0]
            start = elems[1]
            end = elems[2]
            vcf_qry_str = chrm + ":" + start + "-" + end
            out["_".join(elems)] = get_dosages(vcf_map(vcf_qry_str))
    return out


# gets inter LD block correlations
def get_between_block_cors(ld_dosages):
    out = {}
    for v in ld_dosages.keys(): out[v] = {}
    comps = pd.DataFrame(np.triu(np.ones((len(ld_dosages.keys()), len(ld_dosages.keys()))), k = 1),
                         index = ld_dosages.keys(), columns = ld_dosages.keys())
    for ld0 in ld_dosages.keys():
        for ld1 in ld_dosages.keys():
            if comps.loc[ld0, ld1] == 0:
                continue
            out[ld0][ld1] = pd.concat([ld_dosages[ld0], ld_dosages[ld1]]).transpose().corr().loc[ld_dosages[ld0].index, ld_dosages[ld1].index]
    return out


# gets intra LD block correlations
def get_within_block_cors(ld_dosages):
    out = {}
    for ld in ld_dosages.keys():
        out[ld] = ld_dosages[ld].transpose().corr()
    return out

In [ ]:
if __name__ == "__main__":
    chrm = sys.argv[1]
    ld_blocks = sys.argv[2]
    out_base = sys.argv[3]
    f_out_within = open(out_base + "_within", "w+")
    f_out_between = open(out_base + "_between", "w+")
    vcf = "/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC/output/sashimi_plots/all_ROSMAP_samples/vcf_files/ZOD14598_AD_GRM_WGS_2021-04-29_chr%s.recalibrated_variants.leftnorm.filtered.AF.rnaseq.sorted.vcf.gz" % chrm
    vcf_obj = VCF(vcf)
    dosage = get_ld_block_dosages(vcf_obj, ld_blocks)
    for cor in get_within_block_cors(dosage).values():
        print(cor)
        keep = np.triu(np.ones(cor.shape), k=1).astype('bool').reshape(cor.size)
        for r in cor.stack()[keep]:
            f_out_within.write( "%.9f\n" % r)
    f_out_within.close()
    for d in get_between_block_cors(dosage).values():
        for cor in d.values():
            for r in cor.stack():
                f_out_between.write( "%.9f\n" % r)

In [ ]:
#!/bin/bash
#$ -N calc_out
#$ -l h_rt=48:10:00
#$ -P casa
chr=${1}
tmp=(mktemp)
cat ./EUR_LD_blocks.bed | grep -P "chr${chr} " > ${tmp}
python3 ./cor_vcf.py 22 ${tmp} out_chr${chr}


In [3]:
chrm = 1
ld_blocks = '/mnt/vast/hpc/csg/rf2872/Work/pecotmr/tosin_codes/testblock'
out_base = 'out_chr1'
f_out_within = open(out_base + "_within", "w+")
f_out_between = open(out_base + "_between", "w+")

In [4]:
vcf = "/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_DLPFC/output/sashimi_plots/all_ROSMAP_samples/vcf_files/ZOD14598_AD_GRM_WGS_2021-04-29_chr%s.recalibrated_variants.leftnorm.filtered.AF.rnaseq.sorted.vcf.gz" % chrm
vcf_obj = VCF(vcf)

In [5]:
dosage = get_ld_block_dosages(vcf_obj, ld_blocks)

In [6]:
for cor in get_within_block_cors(dosage).values():
        print(cor)
        keep = np.triu(np.ones(cor.shape), k=1).astype('bool').reshape(cor.size)
        for r in cor.stack()[keep]:
            f_out_within.write( "%.9f\n" % r)

                   chr1_16280_T_C  chr1_16841_G_T  chr1_16856_A_G  \
chr1_16280_T_C                NaN             NaN             NaN   
chr1_16841_G_T                NaN        1.000000       -0.060030   
chr1_16856_A_G                NaN       -0.060030        1.000000   
chr1_17005_A_G                NaN       -0.057326        0.006419   
chr1_17147_G_A                NaN        0.661137       -0.028441   
...                           ...             ...             ...   
chr1_2887456_T_G              NaN       -0.031841       -0.017803   
chr1_2887713_C_G              NaN       -0.002087       -0.037630   
chr1_2887908_T_C              NaN        0.012444       -0.007239   
chr1_2887913_CT_C             NaN        0.020557       -0.007239   
chr1_2888245_G_A              NaN        0.026176       -0.017863   

                   chr1_17005_A_G  chr1_17147_G_A  chr1_17408_C_G  \
chr1_16280_T_C                NaN             NaN             NaN   
chr1_16841_G_T          -0.057326

IndexError: Boolean index has wrong length: 34445161 instead of 32954739